# Spark Structured Streaming — Interview Preparation

A comprehensive bank of interview questions with detailed answers and runnable code examples.
Questions are grouped by difficulty so you can warm up before tackling harder problems.

| Section | Difficulty | Topics |
|---------|-----------|--------|
| 1 | Easy | Streaming basics, output modes, triggers, checkpointing |
| 2 | Medium | Windowing, watermarks, joins, stateful ops |
| 3 | Hard | Exactly-once, state stores, Kafka patterns, performance |
| 4 | Expert | Deep internals, advanced patterns, system design |

> **How to use this notebook**
> Read the question, formulate your own answer, then expand the answer cell.
> Run code cells to see the concepts in action.

## Setup

In [ ]:
import tempfile, time, threading
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, expr, window, count, sum as spark_sum, avg,
    current_timestamp, lit, when, rand, concat
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

spark = (
    SparkSession.builder
    .appName('SS-06-Interview-Prep')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
BASE = tempfile.mkdtemp().replace('\\', '/') + '/ss_interview'
print('Temp base:', BASE)

---
## Section 1 — Easy Questions

Core definitions and fundamental concepts.

### Q1. What is Spark Structured Streaming and how does it differ from DStreams?

**Answer:**

**Structured Streaming** is Spark's production-grade streaming engine introduced in Spark 2.0.
It models a streaming data source as an *unbounded table* — new rows arrive continuously.
You write the same DataFrame/SQL API you use for batch, and Spark handles the incremental execution.

| Aspect | DStream (Spark Streaming) | Structured Streaming |
|--------|--------------------------|----------------------|
| API | RDD-based, low-level | DataFrame / Dataset / SQL |
| Fault tolerance | Write-ahead log | Checkpointing + idempotent sources |
| Event time | Manual | Built-in (watermark) |
| Exactly-once | Hard to achieve | Built-in with correct sink |
| Late data | Not handled | Handled via watermarks |
| Status | Maintenance mode | Recommended / active development |

> **Interview tip:** Mention that DStreams operate on *processing time* and have no built-in event-time
> support, whereas Structured Streaming has first-class watermark support for event-time windows.

### Q2. What are the three output modes? When is each valid?

**Answer:**

| Mode | Description | Valid when |
|------|-------------|-----------|
| `append` | Only new rows that will not change are written | Queries with no aggregation, or aggregations with watermark |
| `complete` | The entire result table is rewritten every trigger | Aggregations without watermark |
| `update` | Only rows that changed since the last trigger are written | Aggregations (with or without watermark) |

**Key rule:** `append` mode requires that once a row is output it can **never change**.
That is why aggregations without watermark cannot use `append` — their counts keep growing.
Adding a watermark tells Spark when a window is *finalised*, making `append` safe.

**Trick question:** Can `complete` mode be used with `foreach` sink?
→ Yes, but you must handle the full result set every trigger — not just deltas.

In [ ]:
# Demonstrate output modes with rate source
rate_df = spark.readStream.format('rate').option('rowsPerSecond', 5).load()

# append mode — each row appears once
q_append = (
    rate_df
    .writeStream
    .format('memory')
    .queryName('q_append_demo')
    .outputMode('append')
    .trigger(processingTime='3 seconds')
    .start()
)
time.sleep(7)
print('=== append mode — only new rows each trigger ===')
spark.sql('SELECT * FROM q_append_demo ORDER BY timestamp DESC LIMIT 10').show()
q_append.stop()

# complete mode — full result every trigger
q_complete = (
    rate_df
    .groupBy()
    .count()
    .writeStream
    .format('memory')
    .queryName('q_complete_demo')
    .outputMode('complete')
    .trigger(processingTime='3 seconds')
    .start()
)
time.sleep(7)
print('=== complete mode — total row count rewrites entire table ===')
spark.sql('SELECT * FROM q_complete_demo').show()
q_complete.stop()

### Q3. What is checkpointing and why is it required?

**Answer:**

Checkpointing writes streaming **state** and **progress metadata** to a durable location
(HDFS, S3, ADLS, or local disk for dev) after every micro-batch.

It stores three things:
1. **Offsets** — which source offsets have been processed (for exactly-once replay on restart)
2. **Commit log** — which offsets have been *committed* to the sink
3. **State data** — accumulator values for stateful operations (aggregations, stream-stream joins)

Without checkpointing:
- On driver failure the query restarts from the beginning of the source → duplicate output
- Stateful aggregations lose their accumulated state

```python
query = df.writeStream
    .option('checkpointLocation', '/path/to/checkpoint')
    .format('parquet')
    .start('/path/to/output')
```

> **Interview tip:** The checkpoint directory is tied to a specific query schema and output.
> Changing the query logic or output path requires a **new** checkpoint directory.

### Q4. Explain the four trigger modes.

**Answer:**

| Trigger | How to set | Behaviour |
|---------|-----------|-----------|
| Default (ASAP) | `.trigger()` omitted | Start next micro-batch as soon as the previous one finishes |
| Fixed interval | `.trigger(processingTime='5 seconds')` | Run every N seconds; if processing takes longer, run immediately after |
| Once | `.trigger(once=True)` | Run exactly one micro-batch of all available data, then stop (Spark < 3.3) |
| AvailableNow | `.trigger(availableNow=True)` | Process all available data in multiple batches, then stop (Spark ≥ 3.3, preferred over Once) |

**`once` vs `availableNow`:**
- `once` processes all data in a *single* micro-batch, which can be very large
- `availableNow` processes in *multiple* batches of normal size — better for large backlogs and checkpointing granularity

Common use case for `availableNow`: scheduled batch-like jobs where you want the simplicity
of streaming (watermarks, state) but only run on-demand rather than continuously.

### Q5. What does `isStreaming` tell you and when would you use it?

**Answer:**

`df.isStreaming` returns `True` if the DataFrame represents a streaming source.

Use cases:
- **Guard clauses in helper functions** — prevent accidentally calling a streaming-incompatible
  operation (e.g., `df.show()`, `df.count()`) on a streaming DataFrame
- **Conditional logic** — apply different transformations based on whether the input is streaming or batch

```python
def process(df):
    if df.isStreaming:
        return df.withColumn('processed', lit(True))
    else:
        return df.withColumn('processed', lit(True)).cache()  # batch: cache is OK
```

> **Gotcha:** `df.count()` on a streaming DataFrame raises `AnalysisException`.
> You must use aggregation within `writeStream` instead.

In [ ]:
batch_df  = spark.range(10)
stream_df = spark.readStream.format('rate').option('rowsPerSecond', 1).load()

print('batch_df.isStreaming  :', batch_df.isStreaming)
print('stream_df.isStreaming :', stream_df.isStreaming)

# Safe guard in a helper
def safe_count(df):
    if df.isStreaming:
        return 'N/A — streaming DF; use writeStream aggregation'
    return df.count()

print('batch count  :', safe_count(batch_df))
print('stream count :', safe_count(stream_df))

### Q6. How do you monitor a streaming query?

**Answer:**

Three main approaches:

**1. `query.lastProgress`** — dict with the last micro-batch metrics:
- `numInputRows` — rows processed this batch
- `inputRowsPerSecond` — ingest rate
- `processedRowsPerSecond` — processing throughput
- `durationMs` — time spent in each stage (addBatch, getBatch, etc.)
- `sources[].endOffset` — how far into the source we have read

**2. `query.status`** — current query state (is it active? what is it doing now?)

**3. Spark UI → Structured Streaming tab** — visual timeline of micro-batches

**4. `StreamingQueryListener`** — programmatic hook to push metrics to external systems (Prometheus, etc.)

```python
import time
for _ in range(3):
    time.sleep(5)
    p = query.lastProgress
    if p:
        print(p['numInputRows'], p['processedRowsPerSecond'])
```

---
## Section 2 — Medium Questions

Windowing, watermarks, joins, and stateful operations.

### Q7. What is a watermark? Why is it needed for append mode with aggregations?

**Answer:**

A **watermark** is a moving threshold that tells Spark: *any event with a timestamp older than
(max-seen-event-time − threshold) is considered late and will be dropped*.

```python
df.withWatermark('event_time', '10 minutes')
  .groupBy(window('event_time', '5 minutes'))
  .count()
```

**Why it enables append mode:**
- Without watermark, a window can receive late data forever → its result never finalises → can't use append
- With watermark, once `max_event_time − delay > window_end`, the window is closed
- Spark then emits the final result with `append` and discards the window state

**Why it enables bounded state:**
- Without watermark, Spark must keep state for all windows ever seen
- With watermark, windows older than the watermark are evicted from the state store

**Important:** The watermark advances only when new data arrives with a later event time.
If data stops flowing, the watermark does not advance.

| Mode | Watermark required? |
|------|-------------------|
| `complete` | No (but state grows forever) |
| `update` | No (but state grows forever without it) |
| `append` | **Yes** — needed to know when a window is final |

In [ ]:
import tempfile, os

CK = BASE + '/ck_watermark'
os.makedirs(CK, exist_ok=True)

events = (
    spark.readStream
    .format('rate')
    .option('rowsPerSecond', 10)
    .load()
    .withColumnRenamed('timestamp', 'event_time')
    .withColumnRenamed('value', 'sensor_id')
)

windowed = (
    events
    .withWatermark('event_time', '10 seconds')  # drop data > 10s late
    .groupBy(window('event_time', '5 seconds').alias('win'))
    .count()
)

q = (
    windowed
    .writeStream
    .format('memory')
    .queryName('watermark_demo')
    .outputMode('update')  # update shows windows as they change
    .option('checkpointLocation', CK)
    .trigger(processingTime='5 seconds')
    .start()
)
time.sleep(18)
print('=== Watermarked window counts ===')
spark.sql('SELECT win, count FROM watermark_demo ORDER BY win.start').show(truncate=False)
q.stop()

### Q8. Tumbling windows vs sliding windows — what is the difference?

**Answer:**

| Property | Tumbling | Sliding |
|----------|---------|---------|
| Definition | Fixed, non-overlapping intervals | Fixed duration, slides every N seconds |
| A row belongs to | Exactly one window | One or more windows |
| Syntax | `window(col, '5 min')` | `window(col, '10 min', '5 min')` |
| State size | Proportional to window size | Larger — overlap means more concurrent state |
| Use case | Per-hour totals, daily reports | Rolling averages, moving trends |

**Example (tumbling 10 min):** Events from 10:00–10:10 all land in one bucket.

**Example (sliding 10 min / slide 5 min):** An event at 10:07 lands in both the
10:00–10:10 *and* the 10:05–10:15 windows.

> **Interview gotcha:** Sliding windows with a short slide (e.g., 1-minute slide on a 1-hour window)
> mean each event lands in 60 windows → 60× the state. Always ask 'what slide interval do you actually need?'

### Q9. Explain stream-static join vs stream-stream join.

**Answer:**

**Stream-static join:**
- One side is a static DataFrame (lookup table, dimension table)
- The static side is read once per micro-batch (or broadcast once)
- No watermark required
- Always `inner`, `left outer`, `left anti` — NOT `right outer` or `full outer`
- Best practice: `broadcast()` the static side to avoid shuffle

```python
result = stream_df.join(broadcast(static_df), on='key')
```

**Stream-stream join:**
- Both sides are streaming
- Spark must buffer rows from *both* sides until the matching row arrives
- **Requires watermarks on both sides** to bound the state (otherwise state grows forever)
- Requires a time-range join condition so Spark knows when to evict buffered rows
- Supports inner and (with watermark) left/right outer joins

```python
result = (stream_a.join(stream_b,
    expr('a.key = b.key AND b.time BETWEEN a.time AND a.time + interval 30 seconds')))
```

| Aspect | Stream-Static | Stream-Stream |
|--------|--------------|--------------|
| Watermark | Not needed | Required on both sides |
| State | No state buffered | Both sides buffered |
| Late data handling | Static side is always fresh | Controlled by watermark |
| Supported joins | inner, left, anti | inner, left/right outer (with watermark) |

In [ ]:
# Stream-static join demo
CK2 = BASE + '/ck_join'
os.makedirs(CK2, exist_ok=True)

from pyspark.sql.functions import broadcast

# Static lookup
products = spark.createDataFrame([
    (1, 'Widget'),  (2, 'Gadget'),
    (3, 'Doohickey'),(4, 'Thingamajig'),
], ['product_id', 'product_name'])

orders = (
    spark.readStream.format('rate').option('rowsPerSecond', 4).load()
    .withColumn('order_id', col('value'))
    .withColumn('product_id', ((col('value') % 4) + 1).cast('int'))
    .drop('value')
)

enriched = orders.join(broadcast(products), on='product_id')

q = (
    enriched.writeStream
    .format('memory').queryName('stream_static_join')
    .outputMode('append')
    .option('checkpointLocation', CK2)
    .trigger(processingTime='3 seconds')
    .start()
)
time.sleep(8)
print('=== Stream-static join — orders enriched with product names ===')
spark.sql('SELECT * FROM stream_static_join ORDER BY order_id LIMIT 12').show()
q.stop()

### Q10. What is `foreachBatch` and when would you use it?

**Answer:**

`foreachBatch` is a sink hook that passes each micro-batch to a user-defined function
as a regular **batch DataFrame**. This unlocks the full batch API inside a streaming pipeline.

Signature:
```python
def process_batch(batch_df: DataFrame, batch_id: int) -> None:
    ...

query = df.writeStream.foreachBatch(process_batch).start()
```

**When to use it:**
1. Write to a sink that has no native streaming connector (e.g., PostgreSQL, DynamoDB)
2. Apply batch-only operations (e.g., `df.count()`, Delta merge/upsert, `df.show()`)
3. Write to **multiple sinks** from one streaming plan (call multiple writers inside the function)
4. Conditional logic — only write if the batch is non-empty
5. Add idempotency logic using `batch_id`

> **Idempotency tip:** Use `batch_id` as a deduplication key when writing to JDBC or other
> sinks — on retry, the same `batch_id` will be re-processed, so you can use it in an upsert
> condition to avoid double-writes.

In [ ]:
from pyspark.sql import DataFrame

batch_counts = []  # accumulate results for display

def process_batch(batch_df: DataFrame, batch_id: int):
    n = batch_df.count()
    if n == 0:
        return
    batch_counts.append({'batch_id': batch_id, 'row_count': n})
    # Could write to JDBC, Delta, multiple outputs, etc.
    # Here we just print
    print(f'  Batch {batch_id}: {n} rows')

CK3 = BASE + '/ck_foreach'
os.makedirs(CK3, exist_ok=True)

q = (
    spark.readStream.format('rate').option('rowsPerSecond', 8).load()
    .writeStream
    .foreachBatch(process_batch)
    .option('checkpointLocation', CK3)
    .trigger(processingTime='3 seconds')
    .start()
)
time.sleep(12)
q.stop()
print('\nBatch summary:', batch_counts)

### Q11. How does `dropDuplicates` work in streaming and what state does it maintain?

**Answer:**

`dropDuplicates(cols)` in streaming mode maintains a **state store** of previously seen keys.
When a new row arrives, it checks the state store — if the key was seen before, the row is dropped.

```python
df.withWatermark('event_time', '1 hour')
  .dropDuplicates(['event_id', 'event_time'])
```

**Why watermark is critical:**
- Without watermark, the state store grows forever (every event_id seen is stored)
- With watermark, old keys outside the watermark window are evicted
- The watermark defines the duplicate-detection window: events arriving > watermark delay
  after their event_time are assumed non-duplicate (state for them has been evicted)

**Common pitfall:** Using `dropDuplicates` without a watermark in a long-running job
causes the state store (RocksDB by default) to grow unboundedly → OOM or performance degradation.

| Scenario | Risk |
|----------|------|
| No watermark | State grows forever |
| Watermark too large | More memory used, more late events deduplicated |
| Watermark too small | Late duplicates slip through |

---
## Section 3 — Hard Questions

Exactly-once semantics, state stores, Kafka integration, and performance tuning.

### Q12. What does 'exactly-once' mean in Spark Streaming? How is it achieved?

**Answer:**

**Exactly-once** means every input record produces its effect on the output exactly once —
even if the driver crashes and the query restarts.

Spark achieves exactly-once through the combination of:

**1. Idempotent sources** — sources that can be re-read without duplication:
- Kafka (replay from offset), file source (replay from file list)
- The checkpoint stores the last committed offset; on restart Spark re-reads from there

**2. Idempotent sinks** — sinks that can accept re-writes without duplication:
- File sink (each micro-batch writes to a new, uniquely-named file)
- Delta Lake (transactional commits with `txnAppId` + `txnVersion` dedup)
- Memory sink (in-memory, used for testing only)

**3. Two-phase commit (2PC)** for transactional sinks:
- Spark first writes data in a 'pre-commit' phase
- Only after the commit log is updated does it make the data visible
- On restart, uncommitted micro-batches are rolled back and replayed

**What breaks exactly-once:**
- Using `foreach` with a non-idempotent side effect (e.g., incrementing a counter in Redis)
- Writing to a sink that does not deduplicate on retry
- External state mutations inside `foreachBatch` that are not idempotent

| Source | Exactly-once possible? |
|--------|----------------------|
| Kafka | Yes (track offsets) |
| File source | Yes (track processed files) |
| Rate source | No (synthetic — restarts from 0) |
| Socket source | No (no replay capability) |

### Q13. What is the state store in Structured Streaming? What backends are available?

**Answer:**

The **state store** is the key-value storage layer that persists intermediate stateful
computation (aggregations, dedup keys, stream-stream join buffers) across micro-batches.

**Two backends:**

| Backend | Config | Characteristics |
|---------|--------|----------------|
| HDFS-based (default) | `stateStore.providerClass = HDFSBackedStateStoreProvider` | State loaded entirely into JVM heap each micro-batch; simple but memory-intensive |
| RocksDB | `stateStore.providerClass = RocksDBStateStoreProvider` | State kept on disk (RocksDB) with LRU cache; handles much larger state with lower heap pressure |

**When to switch to RocksDB:**
- State size > executor heap (HDFS-based causes OOM)
- High-cardinality keys (millions of unique keys)
- Long watermark windows requiring large retention

**Config to enable RocksDB:**
```python
spark.conf.set('spark.sql.streaming.stateStore.providerClass',
    'org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider')
```

**State checkpointing:** Both backends snapshot state to the checkpoint directory periodically.
On restart, state is restored from the last snapshot + any delta files.

### Q14. Describe the Kafka + Spark Streaming integration. How do you control throughput?

**Answer:**

```python
df = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', 'broker:9092')
    .option('subscribe', 'my-topic')          # or subscribePattern, assign
    .option('startingOffsets', 'latest')       # or 'earliest' or JSON offset map
    .option('maxOffsetsPerTrigger', 10000)     # throughput control
    .load()
)
```

**Key fields returned:**
| Field | Type | Content |
|-------|------|---------|
| key | binary | Message key |
| value | binary | Message payload (cast to string) |
| topic | string | Topic name |
| partition | int | Kafka partition |
| offset | long | Offset within partition |
| timestamp | timestamp | Kafka event timestamp |

**Throughput controls:**
- `maxOffsetsPerTrigger` — cap total messages consumed per micro-batch
- `minOffsetsPerTrigger` — wait until at least N offsets are available (Spark ≥ 3.3)
- `maxTriggerDelay` — max time to wait for `minOffsetsPerTrigger` before running anyway

**Writing back to Kafka:**
```python
result.writeStream
    .format('kafka')
    .option('kafka.bootstrap.servers', 'broker:9092')
    .option('topic', 'output-topic')
    .option('checkpointLocation', '/ck')
    .start()
```

> **Interview tip:** Mention that the Kafka connector is in a separate package:
> `spark-sql-kafka-0-10`. The value column must be cast to `StringType` or `BinaryType` before writing.

### Q15. How do you tune Spark Streaming performance?

**Answer:**

**Source-side tuning:**
- `maxOffsetsPerTrigger` (Kafka) or `maxFilesPerTrigger` (file) — limit input size per batch
- Increase parallelism by adding more Kafka partitions (Spark creates one task per partition)

**Shuffle tuning:**
- Set `spark.sql.shuffle.partitions` to match your data volume — default 200 is often too many for streaming
- Enable AQE (`spark.sql.adaptive.enabled=true`) to coalesce empty shuffle partitions

**State store tuning:**
- Use RocksDB backend for large state
- Set `spark.sql.streaming.stateStore.rocksdb.changelogCheckpointing.enabled=true` for incremental snapshots
- Tune watermark to evict state promptly

**Trigger tuning:**
- Use `processingTime` trigger with a duration that matches your SLA
- Avoid default ASAP trigger if downstream can't keep up — it will create continuous micro-batches

**Memory:**
- Increase `spark.executor.memory` for large in-memory state
- Use `spark.sql.streaming.aggregation.stateFormatVersion=2` (default in newer Spark) for better state encoding

**Monitoring signals:**
- `processedRowsPerSecond` < `inputRowsPerSecond` → processing is falling behind
- `durationMs.triggerExecution` growing → batches are slow
- Kafka consumer lag growing → not consuming fast enough

### Q16. What happens when a streaming query fails? How does recovery work?

**Answer:**

**Failure modes:**
1. Executor failure — Spark retries the task on another executor; stateful ops re-read from state store checkpoint
2. Driver failure — requires external restart (YARN, Kubernetes, or a watchdog); query restarts from checkpoint

**Recovery process:**
1. On restart, Spark reads the checkpoint directory
2. It finds the last committed offset range
3. It replays the source from those offsets
4. State is restored from the last state snapshot + delta files
5. The micro-batch is re-executed and committed to the sink

**For exactly-once:** If the sink is idempotent (file sink, Delta Lake), the replayed batch
is safe. If the sink is not idempotent (JDBC without dedup), the same rows may appear twice.

**Incompatible schema changes:** Changing the query's output schema or transformation logic
while reusing the same checkpoint can cause `StreamingQueryException: Checkpoint corrupt`.
Always start with a fresh checkpoint when making structural query changes.

> **Interview tip:** In production, always configure a **query restart policy** in your orchestrator
> (e.g., `spark.streaming.stopGracefullyOnShutdown=true`, Databricks Job retry settings).

---
## Section 4 — Expert Questions

Deep internals, advanced patterns, and system design questions.

### Q17. Explain the micro-batch execution model. What is the MicroBatchExecution plan?

**Answer:**

Structured Streaming has two execution engines:

| Engine | Trigger | How it works |
|--------|---------|-------------|
| `MicroBatchExecution` | Default, processingTime, once, availableNow | Repeatedly runs a batch Spark job on a bounded input |
| `ContinuousExecution` | `trigger(continuous='1 second')` | Long-running tasks with epoch checkpointing; low latency but limited operators |

**MicroBatchExecution lifecycle (per batch):**
1. **GetOffset** — ask each source for the latest available offset
2. **Determine input** — compute the offset range to process (last committed → latest)
3. **Construct logical plan** — build the streaming logical plan with the new input range
4. **Execute** — run a normal Spark job (Catalyst → physical plan → RDD execution)
5. **Commit** — write commit log entry; advance the 'committed offset' pointer
6. **State cleanup** — evict state older than watermark

**Why micro-batch latency is bounded by batch duration:**
The minimum latency is the time to execute one micro-batch. For sub-second latency
you need Continuous Processing (experimental) or a different streaming system.

**The Increment Batch ID:** Each micro-batch has a monotonically increasing `batch_id`.
This is what `foreachBatch` exposes and what Delta Lake uses for deduplication.

### Q18. What is the dual-sink pattern and when do you use it?

**Answer:**

The **dual-sink pattern** runs two `writeStream` queries on the same transformed DataFrame,
writing to two different sinks concurrently.

```python
enriched = raw_stream.join(broadcast(products), 'product_id') \
                     .withColumn('line_total', col('qty') * col('price'))

# Sink 1: durable store for historical analysis
q1 = enriched.writeStream.format('parquet') \
    .option('checkpointLocation', '/ck1').start('/data/orders')

# Sink 2: in-memory dashboard with windowed aggregation
q2 = enriched.groupBy(window('event_time', '1 min')).agg(spark_sum('line_total')) \
    .writeStream.format('memory').queryName('dashboard') \
    .option('checkpointLocation', '/ck2').outputMode('complete').start()
```

**Use cases:**
- Write raw events to data lake AND aggregated metrics to a dashboard
- Write to a primary sink AND send alerts to a secondary sink
- Fan-out to multiple teams' data stores from a single ingest pipeline

**Important:**
- Each query has its own checkpoint directory
- Each query is independent — one can fall behind without affecting the other
- The shared DataFrame plan is executed independently for each query (no shared computation cache)
  → Use `foreachBatch` with multiple writes inside if you want to share the execution

In [ ]:
# Dual-sink pattern demo
import os

PARQUET_OUT = BASE + '/dual_parquet'
CK_P = BASE + '/ck_dual_parquet'
CK_D = BASE + '/ck_dual_dashboard'
for d in [PARQUET_OUT, CK_P, CK_D]:
    os.makedirs(d, exist_ok=True)

raw = (
    spark.readStream.format('rate').option('rowsPerSecond', 10).load()
    .withColumnRenamed('timestamp', 'event_time')
    .withColumn('amount', (rand(seed=7) * 100).cast('double'))
    .withColumn('category', when(col('value') % 3 == 0, 'A')
                           .when(col('value') % 3 == 1, 'B')
                           .otherwise('C'))
)

# Sink 1: raw events to Parquet
q_parquet = (
    raw.writeStream
    .format('parquet')
    .option('checkpointLocation', CK_P)
    .option('path', PARQUET_OUT)
    .outputMode('append')
    .trigger(processingTime='4 seconds')
    .start()
)

# Sink 2: windowed aggregation to memory dashboard
q_dashboard = (
    raw.withWatermark('event_time', '8 seconds')
    .groupBy('category', window('event_time', '8 seconds').alias('win'))
    .agg(spark_sum('amount').alias('total'), count('*').alias('cnt'))
    .writeStream
    .format('memory')
    .queryName('dual_dashboard')
    .option('checkpointLocation', CK_D)
    .outputMode('update')
    .trigger(processingTime='4 seconds')
    .start()
)

time.sleep(16)

print('=== Dashboard (windowed by category) ===')
spark.sql('SELECT category, total, cnt FROM dual_dashboard ORDER BY category').show()

import glob
parquet_files = glob.glob(PARQUET_OUT + '/*.parquet')
if parquet_files:
    total_rows = spark.read.parquet(PARQUET_OUT).count()
    print(f'Parquet sink: {len(parquet_files)} file(s), {total_rows} total rows')

q_parquet.stop()
q_dashboard.stop()

### Q19. How would you design a streaming pipeline with late-data handling and SLA guarantees?

**Answer:**

This is a system design question. A strong answer covers:

**1. Event-time vs processing-time:**
- Always use event-time (`withWatermark`) for business aggregations
- Processing-time is only appropriate for operational metrics

**2. Watermark sizing:**
- Set watermark = (p95 or p99 end-to-end latency from event-generation to Spark ingest)
- Too small: late data dropped, inaccurate results
- Too large: increased state, delayed output (windows take longer to close)

**3. SLA on output latency:**
- `trigger(processingTime='N seconds')` where N < SLA
- If batch processing time > trigger interval, batches run back-to-back (no gap)
- Monitor `durationMs.triggerExecution`; alert if it approaches SLA

**4. Late-data audit trail:**
- Fork a second sink that captures dropped late records for reconciliation
- Or use `append` mode: emit a final result per window, then a correction record when late data arrives

**5. Backpressure:**
- Cap `maxOffsetsPerTrigger` so one large spike doesn't cause runaway latency
- Use `minOffsetsPerTrigger` + `maxTriggerDelay` to avoid tiny batches during quiet periods

**6. Fault tolerance SLA:**
- Checkpoint to durable storage with replication (S3, ADLS)
- Use Kubernetes or YARN with automatic driver restart
- Recovery time ≈ time to re-process (last checkpoint → failure point)

**Architecture sketch:**
```
Kafka → [Spark Structured Streaming]
          withWatermark(event_time, '2 min')
          groupBy(window(5 min), category)
          agg(sum, count)
          foreachBatch →
              ├─ Delta Lake upsert (durable, queryable)
              └─ Redis publish (low-latency dashboard feed)
```

### Q20. What is `mapGroupsWithState` / `flatMapGroupsWithState`? When do you use them over window aggregations?

**Answer:**

These are the **arbitrary stateful transformation** APIs in Structured Streaming (Dataset API only).
They let you maintain and update completely custom state — not just sum/count — per group key.

| API | Output cardinality | Timeout support |
|-----|--------------------|----------------|
| `mapGroupsWithState` | Exactly one output row per group per batch | Yes |
| `flatMapGroupsWithState` | Zero or more output rows per group per batch | Yes |

**When to use them over window aggregations:**
- Stateful session detection (user sessions that have no fixed duration)
- Complex event sequences (CEP — detect pattern A followed by B within N seconds)
- Per-user rate limiting or fraud pattern detection
- Machine learning model inference that maintains per-key state
- Any state that cannot be expressed as a simple groupBy + agg

**Timeout support:**
- `GroupStateTimeout.ProcessingTimeTimeout` — evict state after N seconds of inactivity
- `GroupStateTimeout.EventTimeTimeout` — evict state based on watermark and event time

**Output modes:**
- `mapGroupsWithState` requires `update` mode
- `flatMapGroupsWithState` supports `append` or `update`

**Limitation:** Only available in the **Dataset API** (Scala/Java/Python typed), not DataFrame SQL.
This means you must define case classes (Scala) or StructType + encoders (Python).

### Q21. Quick-fire: Short answer questions

Test yourself on these common interview quick-fire questions.

---

**Q: Can you use `df.show()` on a streaming DataFrame?**
A: No — `show()` triggers an immediate action and is not supported on streaming DataFrames.
   Use `writeStream.format('memory')` and then query with `spark.sql()`.

---

**Q: What happens if you change the schema of the source while the query is running?**
A: By default, schema inference is fixed at query start. Adding a new column to a file source
   will cause the new column to be `null` (schema is set). Dropping a column may cause errors.
   Use an explicit schema to avoid surprises.

---

**Q: Can multiple streaming queries share a SparkSession?**
A: Yes. Each query runs independently and is tracked in `spark.streams.active`.
   You can start, stop, and await them independently.

---

**Q: What is `query.awaitTermination()`?**
A: It blocks the main thread until the streaming query stops (either due to `stop()` being called
   or due to an exception). Use `awaitTermination(timeout)` for a non-blocking wait.

---

**Q: What is the file source's advantage over Kafka for prototyping?**
A: No external infrastructure needed — just drop files into a directory.
   The file source requires an **explicit schema** (it will not infer from new files).

---

**Q: In which output mode can you use `dropDuplicates` without a watermark?**
A: `update` or `complete` — but state grows forever. Always pair with a watermark in production.

---

**Q: What does `spark.streams.awaitAnyTermination()` do?**
A: Blocks until ANY active streaming query terminates. Useful when running multiple queries
   and you want to know when the first one fails.

In [ ]:
# Quick-fire demos

# 1. Multiple queries in one SparkSession
q_a = spark.readStream.format('rate').load().writeStream \
    .format('memory').queryName('qa').outputMode('append').start()
q_b = spark.readStream.format('rate').option('rowsPerSecond', 2).load() \
    .writeStream.format('memory').queryName('qb').outputMode('append').start()

print('Active queries:', [q.name for q in spark.streams.active])
time.sleep(4)

# 2. awaitTermination with timeout
q_a.stop()
q_b.stop()
print('Queries stopped. Active:', len(spark.streams.active))

### Q22. Walk through this code and explain what each line does.

```python
events = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', 'broker:9092')
    .option('subscribe', 'clicks')
    .option('maxOffsetsPerTrigger', 50000)
    .load()
    .selectExpr('CAST(value AS STRING) AS raw', 'timestamp AS kafka_ts')
)

from pyspark.sql.functions import from_json
schema = StructType([...])
parsed = events.select(from_json('raw', schema).alias('data'), 'kafka_ts')
              .select('data.*', 'kafka_ts')

result = (
    parsed
    .withWatermark('event_time', '5 minutes')
    .groupBy(window('event_time', '1 minute'), 'page_id')
    .agg(count('*').alias('clicks'), avg('dwell_seconds').alias('avg_dwell'))
)

query = (
    result
    .writeStream
    .foreachBatch(lambda df, id: df.write.format('delta').mode('append').save('/delta/clicks'))
    .option('checkpointLocation', '/ck/clicks')
    .outputMode('update')
    .trigger(processingTime='30 seconds')
    .start()
)
```

**Answer — line by line:**

1. `readStream.format('kafka')` — subscribe to Kafka topic 'clicks'
2. `maxOffsetsPerTrigger=50000` — consume at most 50k messages per micro-batch (backpressure)
3. `CAST(value AS STRING)` — Kafka value is binary; cast to string for JSON parsing
4. `from_json('raw', schema)` — parse JSON string into a struct column using explicit schema
5. `select('data.*', 'kafka_ts')` — flatten struct fields to top-level columns
6. `withWatermark('event_time', '5 minutes')` — drop events > 5 min late; enables state eviction
7. `groupBy(window('event_time', '1 minute'), 'page_id')` — tumbling 1-min windows per page
8. `agg(count, avg)` — clicks per window and average dwell time
9. `foreachBatch(lambda)` — write each micro-batch as a batch DataFrame to Delta Lake
10. `outputMode('update')` — emit only windows that changed this batch (compatible with foreachBatch)
11. `trigger(processingTime='30 seconds')` — run every 30s; output latency ≤ 30s + processing time

> **Why `foreachBatch` instead of a native Delta sink?**
> The `foreachBatch` approach allows adding custom logic (dedup, merge instead of append)
> and is required when using Delta's `MERGE INTO` for upserts.

## Teardown

In [ ]:
import shutil
shutil.rmtree(BASE, ignore_errors=True)
print('Temp directories cleaned up.')
spark.stop()
print('SparkSession stopped.')